# 140 · Audio Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/140-audio-agent/audio_agent_workbook.ipynb)

**What this notebook teaches:** How to build a three-node LangGraph pipeline that transcribes audio with Whisper, classifies the caller's intent with GPT-4o-mini, and routes the ticket to the correct support queue — all from a single audio file.

**Two halves:**
- **Part A (cells 1–4):** Pure Python — run anywhere, no API key needed. You'll trace the routing logic with mock transcripts, see all five intent routes, and learn how the classification prompt is structured.
- **Part B (cells 5–7):** Requires `OPENAI_API_KEY` and an audio file (WAV, MP3, or M4A). Part B skips gracefully if the audio file is missing.

| Concept | Why it matters |
|---------|----------------|
| Whisper `audio.transcriptions.create` | Converts speech to text in one API call — supports 50+ languages |
| Intent classification prompt | JSON-schema-in-prompt forces a typed, enumerable intent field |
| `ROUTING_MAP` lookup | Maps intent strings to queue names — easily extended without changing logic |
| Fence-stripping | `strip("```json")` before `json.loads()` handles LLM markdown wrapping |
| Fail-fast / graceful skip | Part B checks for the audio file and skips cleanly if absent |


In [ ]:
%pip install -q openai python-dotenv langgraph


## Part A · Cell 1 — The Three-Node Pipeline

The workflow maps audio to a support queue in three sequential steps:

```
audio file
    │
    ▼
┌─────────────────────┐
│  transcribe_node    │  Whisper API → plain text transcript
└─────────────────────┘
    │
    ▼
┌─────────────────────┐
│  analyze_node       │  GPT-4o-mini → JSON with intent, urgency, sentiment
└─────────────────────┘
    │
    ▼
┌─────────────────────┐
│  route_node         │  ROUTING_MAP lookup → queue name (no API call)
└─────────────────────┘
    │
    ▼
queue string  (e.g. "billing-queue", "tier1-support")
```

**State object** shared across all three nodes:
```python
class AudioState(TypedDict):
    audio_path: str     # input: path to the audio file
    transcript: str     # after transcribe_node
    analysis: dict      # after analyze_node — {intent, urgency, sentiment, ...}
    queue: str          # after route_node — destination queue
```

**Why keep routing separate from analysis?**  
The `route_node` is a pure dictionary lookup — zero tokens, zero latency. Keeping it separate means you can swap or extend the routing logic without touching the LLM prompt.


In [ ]:
# Part A · Cell 2 — Demo routing logic with mock transcripts (no API)
# Shows all 5 intent routes from ROUTING_MAP

ROUTING_MAP = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "cancellation": "retention-team",
    "general": "general-support",
}

# Mock transcripts — one per intent type
MOCK_TRANSCRIPTS = {
    "billing": (
        "Hi, I received my invoice this month and it looks like I was charged twice "
        "for my subscription. The amount is 89 dollars and I only signed up once. "
        "Can you please look into this and issue a refund?"
    ),
    "technical": (
        "Hello, I'm having trouble logging into the dashboard. Every time I enter "
        "my credentials it gives me an error 403 message. I've tried resetting my "
        "password twice but the problem persists."
    ),
    "complaint": (
        "I am very disappointed with the service I received last week. The agent "
        "I spoke to was rude and my issue was never resolved. I want to file a "
        "formal complaint and speak to a manager."
    ),
    "cancellation": (
        "I would like to cancel my account. I've been a customer for two years but "
        "the pricing has gone up too much and I found a cheaper alternative. "
        "Please process the cancellation before the next billing cycle."
    ),
    "general": (
        "Hi, I just wanted to know what your business hours are and whether "
        "you offer live chat support. I have a quick question about setting "
        "up my profile."
    ),
}


def route(intent: str) -> str:
    """Map intent string to queue name. Mirrors src/tools.py route()."""
    return ROUTING_MAP.get(intent, "general-support")


print("Intent routing demo (no API call needed):")
print("=" * 60)
for intent, transcript in MOCK_TRANSCRIPTS.items():
    queue = route(intent)
    print(f"Intent     : {intent}")
    print(f"Queue      : {queue}")
    print(f"Transcript : {transcript[:80]}...")
    print("-" * 60)

# Show fallback behavior for an unknown intent
unknown_queue = route("unknown_intent")
print(f"\nFallback for 'unknown_intent' → '{unknown_queue}'")


## Part A · Cell 3 — The Classification Prompt Structure

The `analyze_node` sends the transcript text to GPT-4o-mini with a prompt that describes the exact JSON schema to return:

```python
prompt = f"""Analyze this support call transcript and respond with JSON only:
{{
  "intent": "billing|technical|general|complaint|cancellation",
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative"
}}

Transcript: {transcript}"""
```

Key prompt design decisions:

| Field | Type | Why enumerated |
|-------|------|---------------|
| `intent` | enum string | Limited set = routing is deterministic |
| `urgency` | enum string | Three levels cover most triage needs |
| `product_mentioned` | string or null | Nullable — many calls don't name a product |
| `action_required` | free text | Let the model describe the next step in natural language |
| `sentiment` | enum string | Simple three-class — easy to aggregate in dashboards |

**"JSON only" instruction** — appending `respond with JSON only` (without markdown fences) reduces but does not eliminate fence wrapping. The stripping code handles both cases.


In [ ]:
# Part A · Cell 4 — Mock classify_and_extract: parse a simulated LLM response
# Shows the fence-stripping logic and what the output dict looks like

import json

# Simulated LLM responses for each intent (some wrapped in fences, some not)
MOCK_LLM_RESPONSES = {
    "billing": '```json\n{\n  "intent": "billing",\n  "urgency": "medium",\n  "product_mentioned": "subscription",\n  "action_required": "Verify charge history and issue refund for duplicate payment",\n  "sentiment": "negative"\n}\n```',
    "technical": '{\n  "intent": "technical",\n  "urgency": "high",\n  "product_mentioned": "dashboard",\n  "action_required": "Investigate 403 error and reset session token",\n  "sentiment": "neutral"\n}',
    "complaint": '  {"intent": "complaint", "urgency": "high", "product_mentioned": null, "action_required": "Escalate to manager and log formal complaint", "sentiment": "negative"}  ',
    "cancellation": '{"intent": "cancellation", "urgency": "medium", "product_mentioned": null, "action_required": "Offer retention discount before processing cancellation", "sentiment": "negative"}',
    "general": '{"intent": "general", "urgency": "low", "product_mentioned": null, "action_required": "Send FAQ link for business hours and live chat info", "sentiment": "positive"}',
}


def mock_classify_and_extract(transcript: str, intent_hint: str) -> dict:
    """Mirrors src/tools.py classify_and_extract() but uses a simulated response."""
    raw = MOCK_LLM_RESPONSES[intent_hint]
    # Same fence-stripping logic as the real function
    text = raw.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


print("Full pipeline simulation (transcribe → classify → route):")
print("=" * 60)
for intent_hint, transcript in MOCK_TRANSCRIPTS.items():
    analysis = mock_classify_and_extract(transcript, intent_hint)
    queue = route(analysis.get("intent", "general"))

    print(f"\nTranscript: {transcript[:70]}...")
    print(f"Analysis  : {json.dumps(analysis, indent=None)}")
    print(f"Queue     : {queue}")
    print("-" * 60)


---
## Part B · Live API Run

The cells below require:
1. `OPENAI_API_KEY` in your `.env` file
2. An audio file at `/tmp/sample.wav` (WAV, MP3, M4A, or any Whisper-supported format)

**Don't have a sample audio file?** You can create one with macOS `say` or record a short voice message:
```bash
say -o /tmp/sample.aiff "Hi, I was charged twice this month and need a refund right away."
ffmpeg -i /tmp/sample.aiff /tmp/sample.wav
```

Cell 5 checks for both the API key and the audio file. If the audio file is missing, the notebook prints a clear message and skips Parts B cells gracefully — no exception is raised.


In [ ]:
# Part B · Cell 5 — Fail-fast setup (check API key + audio file)
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

AUDIO_PATH = "/tmp/sample.wav"  # change this to your audio file path

api_key = os.environ.get("OPENAI_API_KEY", "")
_READY = False

if not api_key:
    print("SKIP: OPENAI_API_KEY not set. Set it in .env and re-run.")
elif not Path(AUDIO_PATH).exists():
    print(f"SKIP: Audio file not found at '{AUDIO_PATH}'.")
    print("Create a sample file with: say -o /tmp/sample.aiff 'I need a refund'")
    print("Then convert: ffmpeg -i /tmp/sample.aiff /tmp/sample.wav")
else:
    client = OpenAI(api_key=api_key)
    try:
        client.models.list()
        print("Connected to OpenAI API.")
        print(f"Audio file : {AUDIO_PATH} ({Path(AUDIO_PATH).stat().st_size:,} bytes)")
        _READY = True
    except Exception as e:
        print(f"SKIP: Cannot reach OpenAI API: {e}")


In [ ]:
# Part B · Cell 6 — Run the full transcribe → analyze → route workflow

if not _READY:
    print("Skipping — prerequisites not met (see Cell 5 output).")
else:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict

    class AudioState(TypedDict):
        audio_path: str
        transcript: str
        analysis: dict
        queue: str

    def transcribe_node(state: AudioState, config: dict) -> AudioState:
        _client = config["configurable"]["client"]
        with open(state["audio_path"], "rb") as f:
            result = _client.audio.transcriptions.create(model="whisper-1", file=f)
        return {**state, "transcript": result.text}

    def analyze_node(state: AudioState, config: dict) -> AudioState:
        _client = config["configurable"]["client"]
        model = config["configurable"].get("model", "gpt-4o-mini")
        prompt = f"""Analyze this support call transcript and respond with JSON only:
{{
  "intent": "billing|technical|general|complaint|cancellation",
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative"
}}

Transcript: {state['transcript']}"""
        resp = _client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
        )
        text = resp.choices[0].message.content.strip().strip("```json").strip("```").strip()
        try:
            analysis = json.loads(text)
        except json.JSONDecodeError:
            analysis = {"raw": text}
        return {**state, "analysis": analysis}

    def route_node(state: AudioState, config: dict) -> AudioState:
        intent = state["analysis"].get("intent", "general")
        return {**state, "queue": route(intent)}

    graph = StateGraph(AudioState)
    graph.add_node("transcribe", transcribe_node)
    graph.add_node("analyze", analyze_node)
    graph.add_node("route", route_node)
    graph.set_entry_point("transcribe")
    graph.add_edge("transcribe", "analyze")
    graph.add_edge("analyze", "route")
    graph.add_edge("route", END)
    workflow = graph.compile()

    cfg = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

    print(f"Processing: {AUDIO_PATH}\n")
    result = workflow.invoke(
        {"audio_path": AUDIO_PATH, "transcript": "", "analysis": {}, "queue": ""},
        config=cfg,
    )

    print(f"Transcript : {result['transcript']}")
    print(f"Analysis   : {json.dumps(result['analysis'], indent=2)}")
    print(f"\nRouted to  : {result['queue']}")


In [ ]:
# Part B · Cell 7 — Inspect the final state object

if not _READY:
    print("Skipping — prerequisites not met (see Cell 5 output).")
else:
    print("Full AudioState after workflow completion:")
    print("=" * 60)
    # Print all fields except internal ones
    for key, value in result.items():
        if not key.startswith("_"):
            label = key.upper()
            if isinstance(value, dict):
                print(f"{label}:")
                print(json.dumps(value, indent=2))
            else:
                print(f"{label}: {value}")
    print("=" * 60)
    print("\nQueue mapping reference:")
    for intent, queue in ROUTING_MAP.items():
        marker = "<-- this call" if queue == result["queue"] else ""
        print(f"  {intent:15} → {queue} {marker}")
